# 03.02 · PyTorch basics (and its substitutes)

**Chapter:** Deep Learning · **Milestone:** M1

## Goals
- [ ] Tensors, dtypes, devices (CUDA / Apple MPS / CPU)
- [ ] Autograd on tensors; `nn.Module`, `Dataset`, `DataLoader`
- [ ] Write the canonical training loop
- [ ] See the same model in MLX and know where JAX and TensorFlow fit

## Resources
- [PyTorch – Learn the basics](https://pytorch.org/tutorials/beginner/basics/intro.html) — free
- [Dive into Deep Learning (d2l.ai)](https://d2l.ai/) — free
- [MLX docs](https://ml-explore.github.io/mlx/) · [JAX docs](https://docs.jax.dev/) — free

| Framework | Who uses it | Notes |
|---|---|---|
| **PyTorch** | almost all research and open-model code; Hugging Face | default; learn this one deeply |
| **JAX** (+ Flax/Optax) | Google DeepMind, some labs | functional, `jit`/`grad`/`vmap`, great on TPUs |
| **TensorFlow / Keras** | legacy production, mobile (TFLite) | Keras 3 can run on PyTorch/JAX backends |
| **MLX** | Apple Silicon users | NumPy-like, unified memory, fast local LLMs on Mac |

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

## 1. Tensors
Like NumPy arrays, plus: they live on a device and can track gradients.

In [ ]:
t = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print(t.shape, t.dtype, t.device)
print(t @ t.T)  # same matmul rules as 01.03
x_np = t.numpy()  # zero-copy on CPU
t_gpu = t.to(device)  # move to accelerator; ops between tensors must be on the same device
print(t_gpu.device, (t_gpu * 2).sum().item())

for dtype in [torch.float32, torch.bfloat16, torch.float16, torch.int8]:
    print(f"{str(dtype):15} bytes/element: {torch.tensor([], dtype=dtype).element_size()}")

`bfloat16` (2 bytes) is how most LLMs are trained and served; int8/int4 quantization (chapter 13) shrinks further.
A 7B-parameter model in bf16 ≈ 14 GB just for weights.

## 2. Autograd on tensors
`requires_grad=True` tracks operations; `.backward()` fills `.grad` — micrograd (03.01), vectorised.

In [ ]:
w = torch.tensor([1.0, -2.0, 0.5], requires_grad=True)
x = torch.tensor([3.0, 1.0, 4.0])
loss = (w @ x - 1.0) ** 2
loss.backward()
print("grad:", w.grad, "| by hand 2*(w·x-1)*x =", 2 * (w @ x - 1.0).item() * x)

## 3. `nn.Module`, `Dataset`, `DataLoader`
- `nn.Module`: holds parameters (registered automatically when you assign layers as attributes) and defines `forward`.
- `DataLoader`: batches and shuffles a `Dataset`.

In [ ]:
torch.manual_seed(0)
N = 1000
X = torch.randn(N, 3)
true_w, true_b = torch.tensor([2.0, -1.0, 0.5]), 0.3
y = X @ true_w + true_b + 0.1 * torch.randn(N)

loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)


class LinearModel(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.linear = nn.Linear(d_in, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)


model = LinearModel(3).to(device)
print(model, "| params:", sum(p.numel() for p in model.parameters()))

## 4. The training loop
You will write this loop (or see it inside a library) hundreds of times. Memorise its five lines:
**forward → loss → zero_grad → backward → step**.

In [ ]:
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

for epoch in range(5):
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)  # 1. forward
        loss = loss_fn(pred, yb)  # 2. loss
        opt.zero_grad()  # 3. clear old grads
        loss.backward()  # 4. backprop
        opt.step()  # 5. update
        total += loss.item() * len(xb)
    print(f"epoch {epoch}: mse {total / N:.4f}")

print("learned w:", model.linear.weight.data.cpu().numpy().round(3), "b:", round(model.linear.bias.item(), 3))

Inference: wrap in `torch.no_grad()` (no graph tracking → faster, less memory) and call `model.eval()`
(switches off dropout etc., 03.05).

In [ ]:
model.eval()
with torch.no_grad():
    print(model(torch.tensor([[1.0, 1.0, 1.0]], device=device)))  # expect ≈ 2 - 1 + 0.5 + 0.3 = 1.8

## 5. The same thing in MLX
MLX feels like NumPy + function transforms (`mx.grad`, `nn.value_and_grad`). Arrays live in unified memory; computation is lazy
until you call `mx.eval`.

In [ ]:
import platform

if platform.machine() == "arm64" and platform.system() == "Darwin":
    import mlx.core as mx
    import mlx.nn as mnn
    import mlx.optimizers as optim

    Xm, ym = mx.array(X.numpy()), mx.array(y.numpy())
    mlx_model = mnn.Linear(3, 1)
    mlx_opt = optim.SGD(learning_rate=0.1)

    def mlx_loss(m, xb, yb):
        return mnn.losses.mse_loss(m(xb).squeeze(-1), yb)

    loss_and_grad = mnn.value_and_grad(mlx_model, mlx_loss)
    for epoch in range(5):
        perm = mx.array(torch.randperm(N).numpy())
        for i in range(0, N, 64):
            idx = perm[i : i + 64]
            loss, grads = loss_and_grad(mlx_model, Xm[idx], ym[idx])
            mlx_opt.update(mlx_model, grads)
            mx.eval(mlx_model.parameters(), mlx_opt.state)
    print("MLX learned w:", mlx_model.weight, "b:", mlx_model.bias)
else:
    print("MLX needs Apple Silicon; skip")

## 6. JAX, for reference (not installed)
```python
import jax, jax.numpy as jnp
def loss(params, x, y):
    w, b = params
    return jnp.mean((x @ w + b - y) ** 2)
grad_fn = jax.jit(jax.grad(loss))        # gradient is a function transform, compiled with XLA
params = [p - 0.1 * g for p, g in zip(params, grad_fn(params, x, y))]
```
No hidden state: parameters are explicit arguments. Powerful, but most LLM tooling you'll touch is PyTorch.

## Exercises
1. Replace SGD with `torch.optim.AdamW(lr=0.01)`. Faster? Then try lr=1.0 with SGD and watch it diverge.
2. Write a custom `Dataset` class (`__len__`, `__getitem__`) that generates the data on the fly.
3. `uv add jax` and port the training loop to JAX. Which of the three APIs do you prefer?

In [ ]:
# your code here